# Traceability Data Extraction

Questo notebook estrae i record relativi alla "tracciabilità" dal dataset `technology_mapping`.
Sfruttiamo l'elaborazione a chunk e il modulo `multiprocessing` per parallelizzare il lavoro sui 12 core / 24 thread.

**Nota:** Per aggirare l'errore di pickling (`AttributeError: module '__main__' has no attribute 'process_file'`) tipico di Jupyter con `multiprocessing`, la logica pesante e le espressioni regolari sono state estratte nel modulo locale `traceability_worker.py`.

In [1]:
import glob
import os
from multiprocessing import Pool, cpu_count
from traceability_worker import process_file

INPUT_DIR = '../../data/technology_mapping'
OUTPUT_DIR = '../../data/traceability'

os.makedirs(OUTPUT_DIR, exist_ok=True)

def main():
    file_pattern = os.path.join(INPUT_DIR, 'reclassified_multiclass_*.csv')
    files = glob.glob(file_pattern)
    print(f"Trovati {len(files)} file da elaborare.")
    
    args = [(f, OUTPUT_DIR) for f in files]
    
    num_processes = min(len(files), 24) # Massimo 24 thread/processi come specificato
    print(f"Avvio Pool con {num_processes} processi per massimizzare Ryzen 9 9900x...")
    
    with Pool(processes=num_processes) as pool:
        results = pool.map(process_file, args)
        
    print("\n--- Riepilogo ---")
    totale_assoluto = 0
    for filename, matches in results:
        print(f"{filename}: {matches} record estratti.")
        totale_assoluto += matches
    print(f"\nTotale assoluto record estratti: {totale_assoluto}")

if __name__ == '__main__':
    main()


Trovati 12 file da elaborare.
Avvio Pool con 12 processi per massimizzare Ryzen 9 9900x...
[reclassified_multiclass_aiuti_2016.csv] Inizio elaborazione...
[reclassified_multiclass_aiuti_2015.csv] Inizio elaborazione...[reclassified_multiclass_aiuti_2023.csv] Inizio elaborazione...

[reclassified_multiclass_aiuti_2018.csv] Inizio elaborazione...
[reclassified_multiclass_aiuti_2020.csv] Inizio elaborazione...
[reclassified_multiclass_aiuti_2022.csv] Inizio elaborazione...
[reclassified_multiclass_aiuti_2019.csv] Inizio elaborazione...
[reclassified_multiclass_aiuti_2017.csv] Inizio elaborazione...
[reclassified_multiclass_aiuti_2024.csv] Inizio elaborazione...
[reclassified_multiclass_aiuti_2015.csv] Fine elaborazione. Match trovati: 3
[reclassified_multiclass_aiuti_2021.csv] Inizio elaborazione...
[reclassified_multiclass_aiuti_2014.csv] Inizio elaborazione...
[reclassified_multiclass_aiuti_2025.csv] Inizio elaborazione...
[reclassified_multiclass_aiuti_2016.csv] Fine elaborazione. Matc

In [2]:
import pandas as pd
from traceability_worker import get_mask

test_texts = [
    # Veri positivi
    "Tracciabilita - Sicurezza",
    "Mo.Ma.Tra. Movimento Materie Tracciabilita",
    "Team4Cosemtics - Tracciabilita e sicurezza nelle aziende del benessere",
    "Sviluppo di una piattaforma blockchain per la provenienza del prodotto",
    # Falsi positivi (dovrebbero ora fallire)
    "Wearable Technologies - Internet of Things",
    "raccontare la filiera corta ad Arezzo",
    "Produzione integrata di pellet e biochar italiani di qualità in filiera corta",
]

df_test = pd.DataFrame({'TITOLO_PROGETTO': test_texts})
df_test['MATCH'] = get_mask(df_test['TITOLO_PROGETTO'].str.lower())
df_test

,TITOLO_PROGETTO,MATCH
0,Tracciabilita - Sicurezza,True
1,Mo.Ma.Tra. Movimento Materie Tracciabilita,True
2,Team4Cosemtics - Tracciabilita e sicurezza nel...,True
3,Sviluppo di una piattaforma blockchain per la ...,True
4,Wearable Technologies - Internet of Things,False
5,raccontare la filiera corta ad Arezzo,False
6,Produzione integrata di pellet e biochar itali...,False


In [ ]:
import glob
import os
import pandas as pd

# Percorsi
input_dir = '../../data/traceability'
output_dir = os.path.join(input_dir, 'training')
os.makedirs(output_dir, exist_ok=True)

# Trova i file
file_pattern = os.path.join(input_dir, 'traceability_aiuti_*.csv')
csv_files = glob.glob(file_pattern)

print(f"Trovati {len(csv_files)} file da aggregare.")

# Carica e concatena
dfs = []
for file in csv_files:
    try:
        df = pd.read_csv(file, usecols=['TITOLO_PROGETTO', 'DESCRIZIONE_PROGETTO'])
        dfs.append(df)
    except Exception as e:
        print(f"Errore nella lettura di {file}: {e}")

if dfs:
    final_df = pd.concat(dfs, ignore_index=True)
    # Aggiungi colonna Tracciabilita
    final_df['Tracciabilita'] = True
    
    # Salva il file
    output_path = os.path.join(output_dir, 'traceability_training_dataset.csv')
    final_df.to_csv(output_path, index=False)
    print(f"Dataset di training salvato in: {output_path}")
    print(f"Totale righe: {len(final_df)}")
else:
    print("Nessun dato trovato da aggregare.")
